In [ ]:
from pathlib import Path
import os, json, glob

REPO = Path("...../project_repo")
CONFIG = REPO / "configs/config_all_cancers.json"

os.chdir(REPO)

os.environ["OUTPUT_ROOT"] = str(REPO)
os.environ["CACHE_ROOT"] = str(REPO / "results")
os.environ["RUN_ID"] = "tutorial_run"

FOLD_ID = 1
GPU_ID = 0

print("Repo:", REPO)
print("Config:", CONFIG)


In [ ]:
# Optional sanity check: make sure config still loads.

with open(CONFIG) as f:
    cfg = json.load(f)

print("Train/val slides:", len(cfg["data_sources_train_val"]))
print("Test slides:", len(cfg["data_sources_test"]))
print("Test slide IDs:", [x["slide_idx"] for x in cfg["data_sources_test"]])


In [ ]:
# TRAIN
# This command does the data loading, preprocessing/cache creation, training, and epoch evaluation.

!python train.py \
  --config_file configs/config_all_cancers.json \
  --fold_id {FOLD_ID} \
  --gpu_id {GPU_ID}


In [ ]:
# Locate the training run and imputed cache.

RUN_DIR = REPO / "results" / os.environ["RUN_ID"]

impute_dirs = sorted(
    (REPO / "results").glob("imputed_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

IMPUTE_DIR = impute_dirs[0]

print("RUN_DIR:", RUN_DIR)
print("IMPUTE_DIR:", IMPUTE_DIR)
print("Checkpoints:")
for fp in sorted((RUN_DIR / "models").glob("epoch_*_model.pth"))[-5:]:
    print(fp)


In [ ]:
# INFERENCE / EVALUATION
# If labels exist, tools/inference.py computes metrics unless --skip_metrics is passed.

SLIDE_ID = 14
OUT_DIR = RUN_DIR / f"inference_slide{SLIDE_ID}"

!python tools/inference.py \
  --experiment_path {RUN_DIR} \
  --config_file configs/config_all_cancers.json \
  --impute_dir {IMPUTE_DIR} \
  --fold_id {FOLD_ID} \
  --slide_id {SLIDE_ID} \
  --gpu_id {GPU_ID} \
  --output_dir {OUT_DIR} \
  --save_counts_csv


In [ ]:
# Inspect outputs.

import pandas as pd
import json

print("Inference output files:")
for fp in sorted(OUT_DIR.glob("*")):
    print(fp.name)

meta_fp = next(OUT_DIR.glob("*_meta.json"))
with open(meta_fp) as f:
    meta = json.load(f)

print(json.dumps(meta["outputs"], indent=2))
print("Metrics summary:", meta.get("per_gene_pearson_summary"))

expr_fp = Path(meta["outputs"]["pred_expr_scaled_csv"])
pred_expr = pd.read_csv(expr_fp, index_col=0)

display(pred_expr.head())
print(pred_expr.shape)
